# 0. Import

In [73]:
!pip install wandb -q

In [74]:
import json
import wandb
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader

In [75]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

True

# 1. Making the Modality

In [76]:
def load_skeleton(trial_path):

    prediction_dir = Path(trial_path) / "predictions"

    json_files = sorted(prediction_dir.glob("*.json"))

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # One person per frame based on our inspection
        person = data[0]

        keypoints = np.asarray(
            person["keypoints"],
            dtype=np.float32
        )

        frames.append(keypoints)

    if len(frames) == 0:
        return None

    return np.stack(frames)

## 1. Building the training index

In [77]:
def temporal_resample(x, target_frames=64):

    T = x.shape[0]

    if T == target_frames:
        return x.astype(np.float32)

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_frames
    )

    output = np.empty(
        (target_frames, x.shape[1], x.shape[2]),
        dtype=np.float32
    )

    for joint in range(x.shape[1]):
        for coord in range(x.shape[2]):

            output[:, joint, coord] = np.interp(
                new_indices,
                old_indices,
                x[:, joint, coord]
            )

    return output

In [78]:
def normalize_skeleton(x):

    # x: (T, 17, 3)

    x = x.copy()

    # Root-relative coordinates
    root = x[:, 0:1, :]

    x = x - root

    return x

In [79]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["label"],
        groups=train_df["user"]
    )
)

train_split = train_df.iloc[train_idx].reset_index(drop=True)
val_split = train_df.iloc[val_idx].reset_index(drop=True)

print("Train:", len(train_split))
print("Validation:", len(val_split))

print("Train users:")
print(sorted(train_split["user"].unique()))

print("\nValidation users:")
print(sorted(val_split["user"].unique()))

Train: 2238
Validation: 693
Train users:
['user17', 'user18', 'user19', 'user20', 'user21', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Validation users:
['user1', 'user16', 'user2', 'user22']


In [172]:
import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class SkeletonDataset(Dataset):

    def __init__(self, df, sequence_length=64):

        self.df = df.reset_index(drop=True)
        self.sequence_length = sequence_length

    def load_skeleton(self, path):

        predictions_path = Path(path) / "predictions"

        json_files = sorted(predictions_path.glob("*.json"))

        frames = []

        for json_file in json_files:

            with open(json_file, "r") as f:
                data = json.load(f)

            if len(data) == 0:
                continue

            keypoints = data[0]["keypoints"]

            frames.append(keypoints)

        if len(frames) == 0:
            return np.zeros(
                (self.sequence_length, 17, 3),
                dtype=np.float32
            )

        skeleton = np.array(frames, dtype=np.float32)

        return skeleton

    def pad_or_sample(self, skeleton):

        T = skeleton.shape[0]

        if T >= self.sequence_length:

            # Uniform temporal sampling
            indices = np.linspace(
                0,
                T - 1,
                self.sequence_length
            ).astype(int)

            skeleton = skeleton[indices]

        else:

            padding = np.zeros(
                (
                    self.sequence_length - T,
                    skeleton.shape[1],
                    skeleton.shape[2]
                ),
                dtype=np.float32
            )

            skeleton = np.concatenate(
                [skeleton, padding],
                axis=0
            )

        return skeleton

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(row["skeleton"])

        skeleton = self.pad_or_sample(skeleton)

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return X, y

In [91]:
train_dataset = SkeletonDataset(
    train_split.assign(skeleton=train_split["path"]),
    sequence_length=64
)

val_dataset = SkeletonDataset(
    val_split.assign(skeleton=val_split["path"]),
    sequence_length=64
)

print("Train dataset:", len(train_dataset))
print("Val dataset:", len(val_dataset))

Train dataset: 2317
Val dataset: 614


In [123]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

In [122]:
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Train batch X: torch.Size([32, 64, 17, 3])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 3])
Val batch y: torch.Size([32])


In [124]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():

    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)

Input : torch.Size([32, 64, 17, 3])
Output: torch.Size([32, 40])


---

# A. Baseline Model

In [131]:
import torch
import torch.nn as nn


class BaseLSTM(nn.Module):
    def __init__(
        self,
        input_size=17 * 3,
        hidden_size=128,
        num_layers=2,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes)
        )

    def forward(self, x):
        # x: [B, T, 17, 3]

        B, T, J, C = x.shape

        # [B, T, 17, 3]
        #        ↓
        # [B, T, 51]
        x = x.reshape(B, T, J * C)

        # output: [B, T, hidden_size]
        output, (h_n, c_n) = self.lstm(x)

        # Last temporal representation
        x = output[:, -1, :]

        # [B, 40]
        x = self.classifier(x)

        return x

In [132]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BaseLSTM().to(device)

print(model)
print("Device:", device)

BaseLSTM(
  (lstm): LSTM(51, 128, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=128, out_features=40, bias=True)
  )
)
Device: cpu


In [133]:
X, y = next(iter(train_loader))

X = X.to(device)

with torch.no_grad():
    output = model(X)

print("Input :", X.shape)
print("Output:", output.shape)
X, y = next(iter(train_loader))

print("Train batch X:", X.shape)
print("Train batch y:", y.shape)

X, y = next(iter(val_loader))

print("Val batch X:", X.shape)
print("Val batch y:", y.shape)

Input : torch.Size([32, 64, 17, 3])
Output: torch.Size([32, 40])
Train batch X: torch.Size([32, 64, 17, 3])
Train batch y: torch.Size([32])
Val batch X: torch.Size([32, 64, 17, 3])
Val batch y: torch.Size([32])


In [140]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [141]:
wandb.init(
    project="CIUX",
    name="baseline-lstm-skeleton",
    config={
        "model": "LSTM",
        "sequence_length": 64,
        "num_keypoints": 17,
        "coordinates": 3,
        "hidden_size": 128,
        "num_layers": 2,
        "dropout": 0.3,
        "batch_size": 32,
        "learning_rate": 1e-3,
        "epochs": 20,
        "optimizer": "Adam",
    }
)

epoch,▁▃▆█
learning_rate,▁▁▁▁
train/accuracy,▁▇█▆
train/loss,█▃▁▁
val/accuracy,▆█▃▁
val/loss,██▁▂
epoch,4
learning_rate,0.001
train/accuracy,0.11308
train/loss,3.32365
val/accuracy,0.13681


In [142]:
epochs = 10

for epoch in range(epochs):

    # ==========================================================
    # TRAIN
    # ==========================================================

    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for X, y in train_loader:

        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(X)

        loss = criterion(output, y)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * X.size(0)

        predictions = output.argmax(dim=1)

        train_correct += (predictions == y).sum().item()
        train_total += y.size(0)

    train_loss /= train_total
    train_accuracy = train_correct / train_total


    # ==========================================================
    # VALIDATION
    # ==========================================================

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for X, y in val_loader:

            X = X.to(device)
            y = y.to(device)

            output = model(X)

            loss = criterion(output, y)

            val_loss += loss.item() * X.size(0)

            predictions = output.argmax(dim=1)

            val_correct += (predictions == y).sum().item()
            val_total += y.size(0)

    val_loss /= val_total
    val_accuracy = val_correct / val_total


    # ==========================================================
    # PRINT
    # ==========================================================

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )


    # ==========================================================
    # W&B
    # ==========================================================

    wandb.log({
        "epoch": epoch + 1,

        "train/loss": train_loss,
        "train/accuracy": train_accuracy,

        "val/loss": val_loss,
        "val/accuracy": val_accuracy,

        "learning_rate": optimizer.param_groups[0]["lr"]
    })

Epoch 01/10 | Train Loss: 3.4966 | Train Acc: 0.1088 | Val Loss: 3.5151 | Val Acc: 0.1352
Epoch 02/10 | Train Loss: 3.3685 | Train Acc: 0.1139 | Val Loss: 3.5019 | Val Acc: 0.1384
Epoch 03/10 | Train Loss: 3.3291 | Train Acc: 0.1191 | Val Loss: 3.4573 | Val Acc: 0.1384
Epoch 04/10 | Train Loss: 3.2913 | Train Acc: 0.1316 | Val Loss: 3.4063 | Val Acc: 0.1433
Epoch 05/10 | Train Loss: 3.2620 | Train Acc: 0.1290 | Val Loss: 3.4345 | Val Acc: 0.0847
Epoch 06/10 | Train Loss: 3.2264 | Train Acc: 0.1567 | Val Loss: 3.3650 | Val Acc: 0.1221
Epoch 07/10 | Train Loss: 3.2034 | Train Acc: 0.1433 | Val Loss: 3.4540 | Val Acc: 0.0782
Epoch 08/10 | Train Loss: 3.2045 | Train Acc: 0.1351 | Val Loss: 3.3564 | Val Acc: 0.1401
Epoch 09/10 | Train Loss: 3.1185 | Train Acc: 0.1696 | Val Loss: 3.4587 | Val Acc: 0.1401
Epoch 10/10 | Train Loss: 3.2796 | Train Acc: 0.1536 | Val Loss: 3.6201 | Val Acc: 0.1368


In [143]:
wandb.finish()

epoch,▁▂▃▃▄▅▆▆▇█
learning_rate,▁▁▁▁▁▁▁▁▁▁
train/accuracy,▁▂▂▄▃▇▅▄█▆
train/loss,█▆▅▄▄▃▃▃▁▄
val/accuracy,▇▇▇█▂▆▁██▇
val/loss,▅▅▄▂▃▁▄▁▄█
epoch,10
learning_rate,0.001
train/accuracy,0.15365
train/loss,3.27957
val/accuracy,0.13681


---

In [173]:
class SkeletonTestDataset(SkeletonDataset):

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        skeleton = self.load_skeleton(row["skeleton"])

        skeleton = self.pad_or_sample(skeleton)

        X = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        return X, row["trial_id"]

In [145]:
from pathlib import Path

BASE = Path("/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test")

for p in BASE.rglob("SM_test_0001"):
    print("Found:", p)

Found: /kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test/SM_test_0001


In [149]:
TEST_ROOT = p.parent
print(TEST_ROOT)

/kaggle/input/datasets/samasiayushman/small-model-track/Testing/Testing/small_model_track_test-007/small_model_track_test


In [175]:
test_dataset = SkeletonTestDataset(
    test_df,
    sequence_length=64
)

print("Test trials:", len(test_dataset))

Test trials: 405


In [176]:
X, trial_id = test_dataset[0]

print("Trial:", trial_id)
print("Shape:", X.shape)
print("dtype:", X.dtype)

Trial: SM_test_0001
Shape: torch.Size([64, 17, 3])
dtype: torch.float32


In [177]:
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

X, trial_ids = next(iter(test_loader))

print("Batch X:", X.shape)
print("Trial IDs:", trial_ids[:5])

Batch X: torch.Size([32, 64, 17, 3])
Trial IDs: ('SM_test_0001', 'SM_test_0002', 'SM_test_0003', 'SM_test_0004', 'SM_test_0005')


In [178]:
model.eval()

all_predictions = []
all_trial_ids = []

with torch.no_grad():

    for X, trial_ids in test_loader:

        X = X.to(device)

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_trial_ids.extend(trial_ids)

In [179]:
print("Number of predictions:", len(all_predictions))
print("Number of trial IDs:", len(all_trial_ids))

print("\nFirst predictions:")
for trial_id, pred in zip(
    all_trial_ids[:10],
    all_predictions[:10]
):
    print(trial_id, "->", pred)

Number of predictions: 405
Number of trial IDs: 405

First predictions:
SM_test_0001 -> 36
SM_test_0002 -> 34
SM_test_0003 -> 36
SM_test_0004 -> 36
SM_test_0005 -> 36
SM_test_0006 -> 36
SM_test_0007 -> 36
SM_test_0008 -> 36
SM_test_0009 -> 36
SM_test_0010 -> 36


In [180]:
from collections import Counter

prediction_counts = Counter(all_predictions)

print("Predicted classes:")
for label, count in sorted(prediction_counts.items()):
    print(f"{label:2d}: {count}")

Predicted classes:
10: 1
34: 4
36: 400


In [189]:
submission = pd.DataFrame({
    "path": [
        f"small_model_track_test/{trial_id}/"
        for trial_id in all_trial_ids
    ],
    "prediction": all_predictions
})

print(submission.head())
print(submission.shape)

print(submission.head())
print(submission.shape)
submission.to_csv("new_Subs.csv",index=False)
print(submission.head())
print(submission.shape)

                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          34
2  small_model_track_test/SM_test_0003/          36
3  small_model_track_test/SM_test_0004/          36
4  small_model_track_test/SM_test_0005/          36
(405, 2)
                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          34
2  small_model_track_test/SM_test_0003/          36
3  small_model_track_test/SM_test_0004/          36
4  small_model_track_test/SM_test_0005/          36
(405, 2)
                                   path  prediction
0  small_model_track_test/SM_test_0001/          36
1  small_model_track_test/SM_test_0002/          34
2  small_model_track_test/SM_test_0003/          36
3  small_model_track_test/SM_test_0004/          36
4  small_model_track_test/SM_test_0005/          36
(405, 2)
